# EDSR ×3 — 미리 학습된 모델로 바로 확인

흐린 위성사진(10 m)을 3배 선명하게(3.33 m) 만든다. 학습은 하지 않고 완성된 가중치를 쓴다.

셀을 위에서부터 실행하세요. GPU 없어도 됩니다.

## 1. 데이터

In [ ]:
import json, os, urllib.request
import numpy as np, imageio.v2 as imageio, matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
REP  = {'training': 'AOI_Barcelona_10_y0128_x0128', 'validation': 'AOI_Paris_1_6_y0064_x0192'}
TEST = 'incheon_600.png'

def fetch(url, path):
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path

def pair(split, stem):
    """(입력 LR, 정답 HR)"""
    hr = imageio.imread(fetch(f'{BASE}/dataset/{split}/HR/{stem}.png', f'{split}/{stem}.png'))
    lr = imageio.imread(fetch(f'{BASE}/dataset/{split}/LR_bicubic/X3/{stem}x3.png', f'{split}/{stem}_lr.png'))
    return lr, hr

def show(items, title=''):
    """items: [(이름, 입력, 정답 또는 None)]"""
    fig, ax = plt.subplots(2, len(items), figsize=(3.3 * len(items), 7.0), squeeze=False)
    for c, (name, lo, hi) in enumerate(items):
        ax[0, c].imshow(lo); ax[0, c].set_title(f'{name}\ninput {lo.shape[0]}px', fontsize=9)
        if hi is None:
            ax[1, c].text(.5, .5, 'no target', ha='center', va='center', fontsize=11, color='#888')
            ax[1, c].set_facecolor('#f2f2f2')
        else:
            ax[1, c].imshow(hi); ax[1, c].set_title(f'target {hi.shape[0]}px', fontsize=9)
        for r in (0, 1): ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
    if title: fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

WEIGHT = fetch(f'{BASE}/models/01_edsr_x3/checkpoints/edsr_ikonosfull_x3_latest.pt', 'weight.pt')
val_lr, val_hr = pair('validation', REP['validation'])
test_lr = imageio.imread(fetch(f'{BASE}/dataset/test/{TEST}', 'test.png'))

show([('validation (Paris)', val_lr, val_hr),
      ('test (Incheon)', test_lr, None)])
print('validation 은 정답이 있어 점수를 낼 수 있고, test 는 실제 촬영본이라 정답이 없다.')

## 2. 모델 적용

In [ ]:
WEIGHT = 'weight.pt'
import torch, torch.nn as nn

def conv(i, o, k=3): return nn.Conv2d(i, o, k, padding=k // 2)

class MeanShift(nn.Conv2d):
    def __init__(self, rng, sign=-1):
        super().__init__(3, 3, 1)
        self.weight.data = torch.eye(3).view(3, 3, 1, 1)
        self.bias.data = sign * rng * torch.tensor([0.4488, 0.4371, 0.4040])
        for p in self.parameters(): p.requires_grad = False

class ResBlock(nn.Module):
    def __init__(self, n):
        super().__init__(); self.body = nn.Sequential(conv(n, n), nn.ReLU(True), conv(n, n))
    def forward(self, x): return self.body(x) + x

class EDSR(nn.Module):
    def __init__(self, nb=16, nf=64, s=3):
        super().__init__()
        self.sub_mean, self.add_mean = MeanShift(255), MeanShift(255, 1)
        self.head = nn.Sequential(conv(3, nf))
        self.body = nn.Sequential(*[ResBlock(nf) for _ in range(nb)], conv(nf, nf))
        self.tail = nn.Sequential(nn.Sequential(conv(nf, nf * s * s), nn.PixelShuffle(s)), conv(nf, 3))
    def forward(self, x):
        x = self.head(self.sub_mean(x))
        return self.add_mean(self.tail(self.body(x) + x))

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net = EDSR().to(dev).eval()
net.load_state_dict(torch.load(WEIGHT, map_location=dev))

@torch.no_grad()
def upscale(lr):
    t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(dev)
    return net(t).clamp(0, 255).round()[0].cpu().numpy().transpose(1, 2, 0).astype(np.uint8)

print(f'EDSR x3 로드 완료 ({dev})')

## 3. 정량 평가

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

SHAVE = 4
def score(pred, gt):
    a, b = pred[SHAVE:-SHAVE, SHAVE:-SHAVE], gt[SHAVE:-SHAVE, SHAVE:-SHAVE]
    return psnr(b, a, data_range=255), ssim(b, a, data_range=255, channel_axis=2)

import cv2

VAL_API = 'https://api.github.com/repos/BWMIN-Hub/SR_practice/contents/dataset/validation/HR'
with urllib.request.urlopen(VAL_API) as r:
    val_names = sorted(x['name'][:-4] for x in json.load(r))

rows = []
for stem in val_names:
    lr, hr = pair('validation', stem)
    bic = cv2.resize(lr, (hr.shape[1], hr.shape[0]), interpolation=cv2.INTER_CUBIC)
    pb, sb = score(bic, hr)
    pe, se = score(upscale(lr), hr)
    rows.append((stem.rsplit('_y', 1)[0].replace('AOI_', ''), pb, sb, pe, se))

m = np.array([[r[1], r[2], r[3], r[4]] for r in rows]).mean(0)
print(f'{"":18s}{"PSNR":>10s}{"SSIM":>10s}')
print(f'{"Bicubic":18s}{m[0]:10.2f}{m[1]:10.4f}')
print(f'{"EDSR":18s}{m[2]:10.2f}{m[3]:10.4f}')
print(f'{"차이":18s}{m[2]-m[0]:+10.2f}{m[3]-m[1]:+10.4f}   (검증 {len(rows)}장 평균)')

In [ ]:
seen, labels = {}, []
for n, *_ in rows:
    seen[n] = seen.get(n, 0) + 1
    labels.append(n if sum(1 for r in rows if r[0] == n) == 1 else f'{n}-{seen[n]}')
idx, w = np.arange(len(rows)), 0.38

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, (j, k, name) in zip(ax, [(1, 3, 'PSNR (dB)'), (2, 4, 'SSIM')]):
    b, e = [r[j] for r in rows], [r[k] for r in rows]
    a.bar(idx - w/2, b, w, label='Bicubic', color='#9aa5b1')
    a.bar(idx + w/2, e, w, label='EDSR', color='#2f6f9f')
    a.set_xticks(idx); a.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    a.set_title(name); a.set_ylim(min(b + e) * .97, max(b + e) * 1.02)
    a.grid(axis='y', alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. 결과

In [ ]:
def zoom(panels, size=110, title=''):
    """가장 복잡한 구역을 찾아 확대 비교. panels: [(이름, 이미지)]"""
    import cv2
    ref = panels[-1][1]
    e = cv2.Canny(cv2.cvtColor(ref, cv2.COLOR_RGB2GRAY), 50, 150)
    best, bs = (0, 0), -1
    for y in range(0, ref.shape[0] - size, size // 2):
        for x in range(0, ref.shape[1] - size, size // 2):
            v = e[y:y+size, x:x+size].mean()
            if v > bs: best, bs = (y, x), v
    y, x = best
    fig, ax = plt.subplots(1, len(panels), figsize=(2.9 * len(panels), 3.2))
    for a, (n, im) in zip(np.atleast_1d(ax), panels):
        a.imshow(im[y:y+size, x:x+size], interpolation='nearest')
        a.set_title(n, fontsize=9); a.set_xticks([]); a.set_yticks([])
    if title: fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

sr = upscale(val_lr)
bic = cv2.resize(val_lr, (val_hr.shape[1], val_hr.shape[0]), interpolation=cv2.INTER_CUBIC)
zoom([('Bicubic', bic), ('EDSR', sr), ('Target HR', val_hr)], title='validation (Paris)')

t_sr = upscale(test_lr)
t_bic = cv2.resize(test_lr, (t_sr.shape[1], t_sr.shape[0]), interpolation=cv2.INTER_CUBIC)
imageio.imwrite('incheon_sr.png', t_sr)
zoom([('Bicubic', t_bic), ('EDSR', t_sr)], title='test (Incheon) — no target')
print('incheon_sr.png 저장 완료')